# Humanoid Walking with GTDynamics and MuJoCo

This notebook demonstrates a walking humanoid robot using GTDynamics for dynamics computation and MuJoCo for visualization.

The example uses a simple biped robot model and generates a walking trajectory using GTDynamics' multi-phase optimization framework.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gtdynamics as gtd
from gtdynamics import Phase, WalkCycle, Trajectory
import gtsam
from gtsam import Pose3, Point3, Rot3
from gtsam.noiseModel import Isotropic

# MuJoCo imports
try:
    import mujoco
    import mujoco.viewer
    MUJOCO_AVAILABLE = True
except ImportError:
    print("MuJoCo not available. Install with: pip install mujoco")
    MUJOCO_AVAILABLE = False

## 2. Load Humanoid Robot Model

We'll use the biped URDF model which represents a simple humanoid with two legs.

In [ ]:
# Load the biped humanoid robot
robot = gtd.CreateRobotFromFile(gtd.URDF_PATH + "/biped.urdf")

# Print robot information
print(f"Robot name: {robot.name()}")
print(f"Number of links: {len(robot.links())}")
print(f"Number of joints: {len(robot.joints())}")

# List all links
link_names = [(link.id(), link.name()) for link in robot.links()]
link_names.sort()
print("\nLinks:")
for link_id, link_name in link_names:
    print(f"  {link_id}: {link_name}")

## 3. Define Optimization Parameters

Set up noise models and environment parameters for the optimization.

In [ ]:
# Noise models for constraints and objectives
sigma_dynamics = 1e-5    # standard deviation of dynamics constraints
sigma_objectives = 1e-6  # standard deviation of additional objectives

dynamics_model_6 = Isotropic.Sigma(6, sigma_dynamics)
dynamics_model_1 = Isotropic.Sigma(1, sigma_dynamics)
objectives_model_6 = Isotropic.Sigma(6, sigma_objectives)
objectives_model_1 = Isotropic.Sigma(1, sigma_objectives)

# Environment parameters
gravity = np.array([0, 0, -9.8])
mu = 1.0  # friction coefficient

# Create optimizer settings and dynamics graph builder
opt = gtd.OptimizerSetting(sigma_dynamics)
graph_builder = gtd.DynamicsGraph(opt, gravity, None)

print("Optimization parameters configured")

## 4. Define Walking Gait Pattern

Create a walking trajectory with alternating foot contacts.
The biped has two feet: lower0 (right foot) and lower2 (left foot).

In [ ]:
# Define the feet links
left_foot = robot.link("lower0")
right_foot = robot.link("lower2")

print(f"Left foot: {left_foot.name()} (ID: {left_foot.id()})")
print(f"Right foot: {right_foot.name()} (ID: {right_foot.id()})")

# Contact point in the foot's center of mass frame
# The contact is at the tip of the lower leg (sphere at the end)
contact_in_com = np.array([0.28, 0, 0])  # From biped URDF: sphere at xyz="0.28 0 0"

# Define phases for walking gait
# Phase 1: Both feet on ground (double support)
double_support = Phase(15, [left_foot, right_foot], contact_in_com)

# Phase 2: Right foot on ground, left foot swinging
right_support = Phase(20, [right_foot], contact_in_com)

# Phase 3: Both feet on ground (double support)
double_support2 = Phase(15, [left_foot, right_foot], contact_in_com)

# Phase 4: Left foot on ground, right foot swinging
left_support = Phase(20, [left_foot], contact_in_com)

# Create a walk cycle
walk_cycle = WalkCycle()
walk_cycle.addPhase(double_support)
walk_cycle.addPhase(right_support)
walk_cycle.addPhase(double_support2)
walk_cycle.addPhase(left_support)

# Create trajectory with 2 walk cycles
num_cycles = 2
trajectory = Trajectory(walk_cycle, num_cycles)

print(f"\nWalking trajectory created with {num_cycles} cycles")
print(f"Total phases: {trajectory.numPhases()}")
print(f"Final time step: {trajectory.getEndTimeStep(trajectory.numPhases() - 1)}")

## 5. Build Trajectory Optimization Problem

Create the factor graph for multi-phase trajectory optimization.

In [ ]:
# Create multi-phase trajectory factor graph
collocation = gtd.CollocationScheme.Euler
graph = trajectory.multiPhaseFactorGraph(robot, graph_builder, collocation, mu)

print(f"Factor graph created with {graph.size()} factors")

# Build contact point objectives
# Define step size for forward walking
step = np.array([0.15, 0, 0])  # 15 cm forward per step
ground_height = 0.0  # Ground plane at z=0

objectives = trajectory.contactPointObjectives(
    robot, 
    Isotropic.Sigma(3, 1e-7), 
    step, 
    ground_height
)

print(f"Contact point objectives created")

## 6. Add Body Pose Objectives

Constrain the body to maintain upright posture during walking.

In [ ]:
# Get final time step
K = trajectory.getEndTimeStep(trajectory.numPhases() - 1)

# Get body link
base_link = robot.link("body")

# Zero twist (no velocity)
Z_6x1 = np.zeros((6,), float)

# Add body pose objectives for each timestep
# Keep body upright and at a reasonable height
body_height = 0.5  # meters above ground

for k in range(K+1):
    # Target pose: upright orientation, height above ground
    target_pose = Pose3(Rot3(), Point3(0, 0.0, body_height))
    
    objectives.push_back(
        gtd.LinkObjectives(base_link.id(), k)
        .pose(target_pose, Isotropic.Sigma(6, 5e-5))
        .twist(Z_6x1, Isotropic.Sigma(6, 5e-5))
    )

print(f"Body pose objectives added for {K+1} timesteps")

## 7. Add Boundary Conditions

Set initial and final conditions for the trajectory.

In [ ]:
# Add link and joint boundary conditions to factor graph
trajectory.addBoundaryConditions(
    objectives, 
    robot, 
    dynamics_model_6,
    dynamics_model_6, 
    objectives_model_6,
    objectives_model_1, 
    objectives_model_1
)

# Constrain all Phase keys to have duration of 1/240 seconds
dt = 1.0 / 240.0
for phase_idx in range(trajectory.numPhases()):
    objectives.push_back(
        gtd.PriorFactorDouble(
            gtd.PhaseKey(phase_idx), 
            dt, 
            objectives_model_1
        )
    )

print(f"Boundary conditions added")
print(f"Time step dt = {dt} seconds")

## 8. Solve Trajectory Optimization

Combine the dynamics constraints and objectives, then optimize.

In [ ]:
# Combine graph and objectives
graph.push_back(objectives)

print(f"Total factor graph size: {graph.size()} factors")

# Initialize values
init_vals = trajectory.multiPhaseInitialValues(robot, dt)

print(f"Initial values created with {init_vals.size()} variables")

# Optimize
print("\nStarting optimization...")
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, init_vals)
result = optimizer.optimize()

print(f"Optimization complete!")
print(f"Initial error: {graph.error(init_vals):.6e}")
print(f"Final error: {graph.error(result):.6e}")

## 9. Extract and Visualize Results

Extract joint angles and positions from the optimized trajectory.

In [ ]:
# Extract joint angles over time
joint_angles = {}
body_positions = []
times = []

for k in range(K+1):
    # Get body pose
    body_pose = gtd.Pose(result, base_link.id(), k)
    body_positions.append(body_pose.translation())
    times.append(k * dt)
    
    # Get joint angles
    for joint in robot.joints():
        joint_name = joint.name()
        if joint_name not in joint_angles:
            joint_angles[joint_name] = []
        joint_angles[joint_name].append(gtd.JointAngle(result, joint.id(), k))

body_positions = np.array(body_positions)
print(f"Extracted trajectory with {len(times)} timesteps")

# Plot body trajectory
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(times, body_positions[:, 0])
axes[0].set_ylabel('X Position (m)')
axes[0].set_title('Body Position Over Time')
axes[0].grid(True)

axes[1].plot(times, body_positions[:, 1])
axes[1].set_ylabel('Y Position (m)')
axes[1].grid(True)

axes[2].plot(times, body_positions[:, 2])
axes[2].set_ylabel('Z Position (m)')
axes[2].set_xlabel('Time (s)')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"\nBody traveled {body_positions[-1, 0] - body_positions[0, 0]:.3f} meters forward")

In [ ]:
# Plot joint angles
fig, axes = plt.subplots(len(joint_angles), 1, figsize=(10, 2*len(joint_angles)), sharex=True)

if len(joint_angles) == 1:
    axes = [axes]

for idx, (joint_name, angles) in enumerate(sorted(joint_angles.items())):
    axes[idx].plot(times, angles)
    axes[idx].set_ylabel(f'Joint {joint_name}\n(rad)')
    axes[idx].grid(True)

axes[-1].set_xlabel('Time (s)')
axes[0].set_title('Joint Angles Over Time')
plt.tight_layout()
plt.show()

## 10. Visualize with MuJoCo

Use MuJoCo to visualize the walking motion in 3D.

In [ ]:
if MUJOCO_AVAILABLE:
    print("Setting up MuJoCo visualization...")
    
    # Convert URDF to MuJoCo XML format
    # Note: MuJoCo can load URDF directly
    urdf_path = gtd.URDF_PATH + "/biped.urdf"
    
    try:
        # Load model from URDF
        model = mujoco.MjModel.from_xml_path(urdf_path)
        data = mujoco.MjData(model)
        
        print(f"MuJoCo model loaded successfully")
        print(f"  Number of bodies: {model.nbody}")
        print(f"  Number of joints: {model.njnt}")
        print(f"  Number of DOFs: {model.nv}")
        
        # Function to update MuJoCo state from GTDynamics results
        def update_mujoco_state(k):
            """Update MuJoCo data with state from timestep k"""
            # Set joint positions
            joint_idx = 0
            for joint in robot.joints():
                angle = gtd.JointAngle(result, joint.id(), k)
                if joint_idx < model.nq:
                    data.qpos[joint_idx] = angle
                joint_idx += 1
            
            # Forward kinematics
            mujoco.mj_forward(model, data)
        
        # Interactive visualization
        print("\nStarting MuJoCo viewer...")
        print("Note: The viewer will play the walking trajectory in a loop.")
        print("Close the viewer window to continue.")
        
        with mujoco.viewer.launch_passive(model, data) as viewer:
            # Simulation loop
            k = 0
            frame_count = 0
            
            while viewer.is_running():
                # Update state
                update_mujoco_state(k)
                
                # Sync viewer
                viewer.sync()
                
                # Advance timestep
                frame_count += 1
                if frame_count % 2 == 0:  # Skip frames for better visualization speed
                    k = (k + 1) % (K + 1)
        
        print("Viewer closed.")
        
    except Exception as e:
        print(f"Error loading model in MuJoCo: {e}")
        print("This may be due to URDF compatibility issues.")
        print("MuJoCo visualization will be skipped.")
else:
    print("MuJoCo not available.")
    print("To install MuJoCo, run: pip install mujoco")
    print("Then restart the kernel and run this notebook again.")

## 11. Summary

This notebook demonstrated:
1. Loading a humanoid robot model (biped) in GTDynamics
2. Defining a walking gait pattern with alternating foot contacts
3. Setting up trajectory optimization with dynamics constraints
4. Solving for an optimal walking trajectory
5. Visualizing the results with matplotlib plots
6. (Optional) 3D visualization with MuJoCo

The optimization framework in GTDynamics allows for:
- Multi-phase contact dynamics
- Kinodynamic constraints (both kinematics and dynamics)
- Custom objectives and constraints

You can modify this example by:
- Changing the step size or number of walk cycles
- Adjusting the body height or pose targets
- Using different robot models (NAO, Atlas, etc.)
- Adding more complex gait patterns
- Incorporating terrain or obstacles